# 2.3 Demo - Processing Records from CDF and Propagating Changes
**Lessons:** Capturing Changed Data · Deleting Data in Databricks

GDPR Art. 17 and CCPA give people the **right to be forgotten**. In a lakehouse that means:
1. Delete the person from the source-of-truth table
2. **Propagate** the delete to every downstream table (Silver, Gold, feature tables, ...). This demo uses **Change Data Feed (CDF)** to do it incrementally.
3. Make the delete **physical**: Delta keeps old files for time travel, and deletion vectors only *mark* rows as deleted.

![CRM CDC file and delete_requests are merged into users_bronze with CDF on; a readChangeFeed stream with a checkpoint runs apply_changes into users_silver and events_silver; REORG PURGE plus VACUUM removes the data physically](../images/2.3-delete-propagation.svg)

**How to read it:** sections A and B build the middle (Bronze with CDF, then the stream into Silver), C and E feed the blue boxes into Bronze, and G is the red box. Every `propagate_user_changes()` call is one run of the green box, starting from the checkpoint.

In [ ]:
from adelab.classroom import setup
from adelab import datagen
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

DA = setup("2.3", reset=True)
USERS, EVENTS, UPDATES = (f"{DA.paths.landing}/{d}" for d in ("users", "events", "updates"))
datagen.land_users(USERS)
datagen.land_user_events(EVENTS)

## A. Bronze with Change Data Feed
`delta.enableChangeDataFeed = true` makes every commit also record **row-level changes** with `_change_type` (`insert`, `update_preimage`, `update_postimage`, `delete`), `_commit_version`, and `_commit_timestamp`. Deletion vectors make deletes cheap: no file rewrite, just a bitmap.

In [ ]:
%%sql
CREATE TABLE users_bronze (
  user_id BIGINT, first_name STRING, last_name STRING, email STRING, phone STRING, ssn STRING,
  dob DATE, gender STRING, street STRING, city STRING, zip STRING, region STRING, salary BIGINT,
  signup_ts TIMESTAMP, updated_ts TIMESTAMP)
TBLPROPERTIES (delta.enableChangeDataFeed = true, delta.enableDeletionVectors = true)

In [ ]:
(spark.read.json(USERS)
      .withColumn("dob", F.to_date("dob")).withColumn("signup_ts", F.to_timestamp("signup_ts"))
      .withColumn("updated_ts", F.col("signup_ts"))
      .select(*spark.table("users_bronze").columns)
      .write.mode("append").saveAsTable("users_bronze"))

spark.read.json(EVENTS).withColumn("event_ts", F.to_timestamp("event_ts")).write.saveAsTable("events_silver")
print(spark.table("users_bronze").count(), "users |", spark.table("events_silver").count(), "events")

## B. Silver: pseudonymized users, maintained from the change feed
`propagate_user_changes()` reads **only the changes since the last run** (a checkpoint tracks the last processed version) and applies them:
- `insert` / `update_postimage` → upsert the pseudonymized row into `users_silver`
- `delete` → delete from `users_silver` **and** delete the person's rows from `events_silver`

Within one micro-batch a user can change several times, so keep only the **latest** change per user (highest `_commit_version`, with deletes winning ties).

In [ ]:
SALT = "demo-only-salt"   # on Databricks: dbutils.secrets.get("privacy", "pii_salt")

spark.sql("""CREATE TABLE IF NOT EXISTS users_silver (
  user_id BIGINT, user_key STRING, age_band STRING, city STRING, region STRING, gender STRING,
  signup_ts TIMESTAMP, updated_ts TIMESTAMP)""")

def apply_changes(batch_df, batch_id):
    changes = batch_df.filter("_change_type != 'update_preimage'")
    latest = (changes
        .withColumn("rn", F.row_number().over(Window.partitionBy("user_id").orderBy(
            F.col("_commit_version").desc(), (F.col("_change_type") == "delete").desc())))
        .filter("rn = 1")
        .select("user_id", "_change_type",
                F.sha2(F.concat(F.lit(SALT), F.lower("email")), 256).alias("user_key"),
                F.concat((F.floor(F.months_between(F.current_date(), "dob") / 120) * 10).cast("int"),
                         F.lit("s")).alias("age_band"),
                "city", "region", "gender", "signup_ts", "updated_ts"))

    (DeltaTable.forName(spark, "users_silver").alias("t")
        .merge(latest.alias("s"), "t.user_id = s.user_id")
        .whenMatchedDelete(condition="s._change_type = 'delete'")
        .whenMatchedUpdateAll(condition="s._change_type != 'delete'")
        .whenNotMatchedInsertAll(condition="s._change_type != 'delete'")
        .execute())

    deleted = latest.filter("_change_type = 'delete'").select("user_id")
    (DeltaTable.forName(spark, "events_silver").alias("e")
        .merge(deleted.alias("d"), "e.user_id = d.user_id")
        .whenMatchedDelete()
        .execute())

def propagate_user_changes():
    (spark.readStream
          .option("readChangeFeed", "true")
          .table("users_bronze")
          .writeStream
          .foreachBatch(apply_changes)
          .option("checkpointLocation", f"{DA.paths.checkpoints}/users_cdf")
          .trigger(availableNow=True)
          .start().awaitTermination())
    print("silver users:", spark.table("users_silver").count(), "| events:", spark.table("events_silver").count())

propagate_user_changes()

## C. Apply a batch of updates and deletes from the operational system
The CDC file has `operation` = `UPDATE` / `DELETE`. Apply it to Bronze with one `MERGE`.

In [ ]:
datagen.land_user_updates(UPDATES, batch=0)
updates = (spark.read.json(UPDATES)
           .withColumn("dob", F.to_date("dob")).withColumn("signup_ts", F.to_timestamp("signup_ts"))
           .withColumn("updated_ts", F.to_timestamp("updated_ts")))
display(updates.groupBy("operation").count())

(DeltaTable.forName(spark, "users_bronze").alias("t")
    .merge(updates.alias("s"), "t.user_id = s.user_id")
    .whenMatchedDelete(condition="s.operation = 'DELETE'")
    .whenMatchedUpdate(condition="s.operation = 'UPDATE'",
                       set={c: f"s.{c}" for c in spark.table("users_bronze").columns if c != "user_id"})
    .execute())

## D. Look at the change feed
`table_changes('table', start_version[, end_version])` in SQL, or `spark.read.option("readChangeFeed", "true")` in Python. An update produces a **pre-image and a post-image**. A delete produces one `delete` row that carries the **full deleted record**.

![A MERGE that updates user 7 and deletes user 9 creates version 2 and three change rows: update_preimage, update_postimage and delete, all with _commit_version 2](../images/2.3-cdf-change-rows.svg)

**How to read it:** the MERGE in section C was this commit, only with more users. The query below returns the bottom table for it, and `apply_changes()` skips the pre-images and uses the `delete` rows to remove the person downstream.

In [ ]:
%%sql
SELECT _change_type, _commit_version, user_id, email, city
FROM table_changes('users_bronze', 2)
ORDER BY user_id, _commit_version, _change_type

In [ ]:
propagate_user_changes()

In [ ]:
deleted_ids = [r.user_id for r in updates.filter("operation = 'DELETE'").select("user_id").collect()]
print("deleted user_ids:", deleted_ids)
print("still in silver :", spark.table("users_silver").filter(F.col("user_id").isin(deleted_ids)).count())
print("still in events :", spark.table("events_silver").filter(F.col("user_id").isin(deleted_ids)).count())

## E. A formal "right to be forgotten" request
Track requests in a table so you can prove when each one was completed (auditors ask for this).

In [ ]:
req_ids = [r.user_id for r in spark.table("users_bronze").orderBy("user_id").limit(2).collect()]
print("deletion requested for user_ids", req_ids)

In [ ]:
spark.sql("CREATE TABLE delete_requests (user_id BIGINT, requested_at TIMESTAMP, status STRING, completed_at TIMESTAMP)")
spark.sql(f"""INSERT INTO delete_requests VALUES
              ({req_ids[0]}, current_timestamp(), 'pending', NULL),
              ({req_ids[1]}, current_timestamp(), 'pending', NULL)""")
# MERGE ... WHEN MATCHED THEN DELETE works everywhere. Open-source Delta does not allow a subquery
# in DELETE ... WHERE user_id IN (SELECT ...), which Databricks accepts.
spark.sql("""MERGE INTO users_bronze t
             USING (SELECT user_id FROM delete_requests WHERE status = 'pending') r
             ON t.user_id = r.user_id
             WHEN MATCHED THEN DELETE""")
display(spark.table("delete_requests"))

In [ ]:
propagate_user_changes()
spark.sql("""UPDATE delete_requests SET status = 'completed', completed_at = current_timestamp()
             WHERE status = 'pending'""")
display(spark.table("delete_requests"))

## F. ...but the data is NOT gone yet
Delta time travel still returns the deleted person. So do the CDF `delete` records, and the files with deletion-vector markers.

![Matrix of where deleted data is still readable after DELETE, after REORG PURGE and after VACUUM: current version, old data files, time travel and change feed; only VACUUM clears the last three](../images/2.3-physical-deletion.svg)

**How to read it:** you are in column 1 now, and the next cell proves the two red cells for time travel and the change feed. Section G runs columns 2 and 3; its last cell checks that time travel fails.

In [ ]:
ids = ", ".join(map(str, req_ids))
display(spark.sql(f"""
  SELECT 'time travel v1' AS source, user_id, email FROM users_bronze VERSION AS OF 1 WHERE user_id IN ({ids})
  UNION ALL
  SELECT 'change feed', user_id, email FROM table_changes('users_bronze', 0)
  WHERE user_id IN ({ids}) AND _change_type = 'delete'"""))

In [ ]:
%%sql
DESCRIBE HISTORY users_bronze

## G. Make the delete physical
1. `REORG TABLE ... APPLY (PURGE)` rewrites files that have deletion vectors, so deleted rows are removed from the **current** files.
2. `VACUUM` removes data files (and CDF `_change_data` files) no longer referenced by versions inside the retention window.

The default retention is 7 days. That is fine for compliance, because GDPR gives you one month, and it protects concurrent readers. **For the demo only** we disable the safety check and vacuum immediately.

In [ ]:
spark.sql("REORG TABLE users_bronze APPLY (PURGE)")
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
display(spark.sql("VACUUM users_bronze RETAIN 0 HOURS"))
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

In [ ]:
try:
    spark.sql(f"SELECT * FROM users_bronze VERSION AS OF 1 WHERE user_id = {req_ids[0]}").show()
except Exception as e:
    print("Time travel to the old version now fails:", type(e).__name__, str(e).splitlines()[0][:150])

## H. Production checklist
- Enable CDF on every table that feeds others, and propagate deletes downstream **incrementally** (as above, or with SDP `AUTO CDC` + `apply_as_deletes`).
- Use **lineage** to find all downstream tables of a PII column, including ML features and exports.
- Schedule `REORG ... PURGE` and `VACUUM` (or rely on **predictive optimization**) so physical deletion happens within your SLA.
- Streaming tables reading from a table with deletes need `skipChangeCommits` (or a CDF reader). Otherwise the stream fails on the non-append commit.
- Keep evidence: the request table plus `DESCRIBE HISTORY` show what was deleted and when.

In [ ]:
DA.cleanup()